# Notebook 02: I-V Curve Simulation

**Objective:** Simulate MOSFET I-V characteristics using Level-1 and Level-3 SPICE models.

We cover:
1. Level-1: Shichman-Hodges square-law model
2. Level-3: Short-channel model with DIBL, v_sat, subthreshold
3. Transfer curves (Id-Vg) and output curves (Id-Vd)
4. Transconductance (Gm) and output conductance (Gds)

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

from src.device.mosfet import (
    MOSFETLevel1, MOSFETLevel3,
    MOSFETParamsLevel1, MOSFETParamsLevel3
)
from src.device.curves import (
    generate_iv_curves, classify_region, OperatingRegion, extract_vth_linear
)
from src.viz.plots import plot_iv_curves, plot_gm

## 1. Level-1 Model: Transfer Characteristics

In [2]:
params_l1 = MOSFETParamsLevel1(W=10e-6, L=0.18e-6, VTH0=0.45)
model_l1 = MOSFETLevel1(params_l1)

id_vg, id_vd = generate_iv_curves(model_l1)

# Plot
fig = plot_iv_curves(id_vg, id_vd, "Level-1 MOSFET (Shichman-Hodges)")
plt.show()

C:\Users\11519\AppData\Local\Temp\ipykernel_25260\2731643052.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Level-3 Model: Short-Channel Effects

In [3]:
params_l3 = MOSFETParamsLevel3(W=10e-6, L=0.18e-6, VTH0=0.45)
model_l3 = MOSFETLevel3(params_l3)

id_vg_l3, id_vd_l3 = generate_iv_curves(model_l3)

fig = plot_iv_curves(id_vg_l3, id_vd_l3, "Level-3 MOSFET (Short-Channel)")
plt.show()

C:\Users\11519\AppData\Local\Temp\ipykernel_25260\1237081740.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Level-1 vs Level-3 Comparison

Key difference: Level-3 has subthreshold current and velocity saturation.

In [4]:
vgs = np.linspace(-0.3, 2.5, 200)
vds = 1.0
ids_l1 = model_l1.ids(vgs, vds)
ids_l3 = model_l3.ids(vgs, vds)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.semilogy(vgs, np.maximum(ids_l1, 1e-12), label='Level-1')
plt.semilogy(vgs, np.maximum(ids_l3, 1e-12), label='Level-3')
plt.xlabel('Vgs (V)'); plt.ylabel('Id (A)')
plt.title(f'Id-Vg Comparison (Vds={vds}V)')
plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
gm_l1 = np.gradient(ids_l1, vgs) * 1e3
gm_l3 = np.gradient(ids_l3, vgs) * 1e3
plt.plot(vgs, gm_l1, label='Level-1')
plt.plot(vgs, gm_l3, label='Level-3')
plt.xlabel('Vgs (V)'); plt.ylabel('Gm (mS)')
plt.title('Transconductance Comparison')
plt.legend(); plt.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

C:\Users\11519\AppData\Local\Temp\ipykernel_25260\1005483805.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Operating Region Analysis

In [5]:
VGS, VDS = np.meshgrid(np.linspace(0, 2.5, 100), np.linspace(0, 2.5, 100))
regions = classify_region(model_l3, VGS, VDS)

region_names = {0: 'Cutoff', 1: 'Linear', 2: 'Saturation', 3: 'Subthreshold'}
colors = {0: 'lightgray', 1: 'lightblue', 2: 'lightcoral', 3: 'lightyellow'}

plt.figure(figsize=(7, 5))
for r, name in region_names.items():
    mask = regions == r
    plt.scatter(VGS[mask], VDS[mask], c=colors[r], s=1, label=name, alpha=0.5)

plt.xlabel('Vgs (V)'); plt.ylabel('Vds (V)')
plt.title('MOSFET Operating Regions')
plt.legend(markerscale=10); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

C:\Users\11519\AppData\Local\Temp\ipykernel_25260\512992845.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Threshold Voltage Extraction

Linear extrapolation method: find max-Gm point, extrapolate to x-intercept.

In [6]:
vth_extracted = extract_vth_linear(model_l3, np.linspace(0, 2, 200))
print(f"True VTH0:       {params_l3.VTH0:.4f} V")
print(f"Extracted VTH:   {vth_extracted:.4f} V")
print(f"Error:           {abs(vth_extracted - params_l3.VTH0)*1000:.2f} mV")

True VTH0:       0.4500 V
Extracted VTH:   0.4660 V
Error:           15.96 mV


## Summary

- **Level-1** captures long-channel behavior: square-law Id, sharp cutoff
- **Level-3** adds short-channel physics: subthreshold leakage, velocity saturation, DIBL
- **Operating regions** are classified by (Vgs-Vth) vs Vds relationship
- **VTH extraction** via linear extrapolation is accurate to ~10mV

Next: [Notebook 03 — Parameter Extraction](03_parameter_extraction.ipynb)